# Kaggle Full Verdict Runner

Runs full compile+test evaluation on Kaggle runtime and exports machine-readable evidence artifacts.

In [ ]:
WORKDIR = "/kaggle/working/codemod-eval"
WORKFLOW_REPO = "https://github.com/PRADDZY/codemod-v5.git"
WORKFLOW_DIR = f"{WORKDIR}/codemod-v5"
RUN_MODE = "full"
MEMORY_TIERS = "4096,6144,8192,12288"
RESULT_DIR = f"{WORKDIR}/.codemod-eval-final"

print("RUN_MODE:", RUN_MODE)
print("MEMORY_TIERS:", MEMORY_TIERS)
print("RESULT_DIR:", RESULT_DIR)

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

import pandas as pd


def run(cmd, cwd=None, env=None, check=True, capture=False):
    print("$", " ".join(cmd))
    result = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        text=True,
        capture_output=capture,
    )
    if capture:
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(cmd)}")
    return result


api_key = os.environ.get("CODEMOD_API_KEY", "").strip()
if not api_key:
    raise RuntimeError("Missing CODEMOD_API_KEY. Add it as a Kaggle Secret env var before running.")
print("CODEMOD_API_KEY loaded from environment.")

In [ ]:
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)

run(["apt-get", "update"])
run([
    "apt-get",
    "install",
    "-y",
    "git",
    "curl",
    "ca-certificates",
    "build-essential",
    "python3",
])

# Force Node 16 for Hardhat-heavy repositories.
run(["bash", "-lc", "npm install -g n"])
run(["bash", "-lc", "n 16.20.2"])
os.environ["PATH"] = f"/usr/local/bin:{os.environ.get('PATH', '')}"

run(["git", "clone", WORKFLOW_REPO, WORKFLOW_DIR])
run(["npm", "ci"], cwd=WORKFLOW_DIR)

run(["bash", "-lc", "curl -fsSL https://foundry.paradigm.xyz | bash"])
run([
    "bash",
    "-lc",
    "export PATH=\"$HOME/.foundry/bin:/root/.foundry/bin:$PATH\"; foundryup; forge --version",
])

run(["node", "--version"])
run(["npm", "--version"])


In [ ]:
env = os.environ.copy()
env["CODEMOD_API_KEY"] = api_key

result = run(
    [
        "node",
        "./scripts/full-matrix-evaluate.js",
        "--mode",
        RUN_MODE,
        "--workdir",
        RESULT_DIR,
        "--memory-tiers",
        MEMORY_TIERS,
    ],
    cwd=WORKFLOW_DIR,
    env=env,
    check=False,
    capture=True,
)

print("Matrix runner exit code:", result.returncode)
if result.returncode not in (0, 3):
    raise RuntimeError("Matrix execution failed unexpectedly. Check stdout/stderr above.")

In [ ]:
summary_path = Path(RESULT_DIR) / "verdict-summary.json"
artifact_path = Path(RESULT_DIR) / "runs-evidence.tar.gz"

if not summary_path.exists():
    raise RuntimeError(f"Missing summary file: {summary_path}")

summary = json.loads(summary_path.read_text(encoding="utf-8"))
df = pd.DataFrame(summary.get("targets", []))
display(df)

print("Summary:", summary_path)
print("Artifact:", artifact_path)
print("Artifact exists:", artifact_path.exists())

if artifact_path.exists():
    print("Download from Kaggle output files panel:", artifact_path)